In [11]:
import pandas as pd
import numpy as numpy

# loading datasets
df_flights = pd.read_csv("../data/processed/cleaned_flights.csv")
df_hotels = pd.read_csv("../data/processed/cleaned_hotels.csv")
df_trends = pd.read_csv("../data/processed/cleaned_trends.csv")

# turn dates into date objects
df_flights["date_collected"] = pd.to_datetime(df_flights["datetime_collected"]).dt.date
df_hotels["date_collected"] = pd.to_datetime(df_hotels["datetime_collected"]).dt.date
# (trends is a special case since its a UNIX timestamp)
df_trends["date"] = pd.to_datetime(df_trends["timestamp"]).dt.date

# ensure that the values are numeric
df_flights["price"] = pd.to_numeric(df_flights["price"], errors="coerce")
df_hotels["price_php"] = pd.to_numeric(df_hotels["price_php"], errors="coerce")
df_trends["value"] = pd.to_numeric(df_trends["value"], errors="coerce")

In [12]:
# aggregate data by day of collection, seperated by target date and rolling data

fest_flights = df_flights[df_flights["collection_mode"] == "festival_target"]
rolling_flights = df_flights[df_flights["collection_mode"] == "rolling"]

daily_fest_flights = fest_flights.groupby("date_collected").agg(
    flight_fest_min_price=("price", "min"),
    flight_fest_max_price=("price", "max"),
    flight_fest_median_price=("price", "median"),
    flight_fest_mean_price=("price", "mean"),
    flight_fest_quote_count=("price", "count")
).reset_index()

daily_rolling_flights = rolling_flights.groupby("date_collected").agg(
    flight_rolling_min_price=("price", "min"),
    flight_rolling_max_price=("price", "max"),
    flight_rolling_median_price=("price", "median"),
    flight_rolling_mean_price=("price", "mean"),
    flight_rolling_quote_count=("price", "count")
).reset_index()

daily_flights = daily_fest_flights.merge(daily_rolling_flights, on="date_collected", how="outer")

fest_hotels = df_hotels[df_hotels["collection_mode"] == "festival_target"]
rolling_hotels = df_hotels[df_hotels["collection_mode"] == "rolling"]

daily_fest_hotels = fest_hotels.groupby("date_collected").agg(
    hotel_fest_min_price=("price_php", "min"),
    hotel_fest_max_price=("price_php", "max"),
    hotel_fest_median_price=("price_php", "median"),
    hotel_fest_mean_price=("price_php", "mean"),
    hotel_fest_quote_count=("price_php", "count")
).reset_index()

daily_rolling_hotels = rolling_hotels.groupby("date_collected").agg(
    hotel_rolling_min_price=("price_php", "min"),
    hotel_rolling_max_price=("price_php", "max"),
    hotel_rolling_median_price=("price_php", "median"),
    hotel_rolling_mean_price=("price_php", "mean"),
    hotel_rolling_quote_count=("price_php", "count")
).reset_index()

daily_hotels = daily_fest_hotels.merge(daily_rolling_hotels, on="date_collected", how="outer")

daily_trends = df_trends.groupby("date").agg(
    trend_search_score=("value", "max")
).reset_index().rename(columns={"date":"date_collected"})

In [13]:
# make the master_df and sort by date collected (which should be a datetime obj)

master_df = daily_trends.merge(daily_flights, on="date_collected", how="inner")
master_df = master_df.merge(daily_hotels, on="date_collected", how="left")
master_df = master_df.sort_values("date_collected").reset_index(drop=True)

In [14]:
# feature engineering and derived metrics, apparently

In [15]:
master_df["flight_price_premium"] = master_df["flight_fest_median_price"] / master_df["flight_rolling_median_price"]
master_df["hotel_price_premium"] = master_df["hotel_fest_median_price"] / master_df["hotel_rolling_median_price"]

In [16]:
# trend velocity
master_df["trend_velocity_7d"] = master_df["trend_search_score"].pct_change(periods=7)

In [17]:
# lagg time to surge
master_df["trend_score_lag_1d"] = master_df["trend_search_score"].shift(1)
master_df["trend_score_lag_2d"] = master_df["trend_search_score"].shift(2)
master_df["trend_score_lag_3d"] = master_df["trend_search_score"].shift(3)

In [18]:
# price velocity
master_df["flight_median_change_1d"] = master_df["flight_fest_median_price"].diff(1)
master_df["hotel_median_change_1d"] = master_df["hotel_fest_median_price"].diff(1)

In [19]:
# lead time
dates = pd.to_datetime(master_df["date_collected"])
festival_start = pd.to_datetime("2026-10-09")

master_df["days_to_festival"] = (festival_start - dates).dt.days

In [20]:
# missing values handling and saving
from pathlib import Path

output_path = Path("../data/processed/masskara/analysis/master.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
master_df.to_csv(output_path, index=False)